# 7장. RAG 시스템 평가 — 07. 모델 비교 평가 (Model Comparison)

## 학습 목표

- 여러 LLM을 동일한 데이터셋으로 비교 평가
- LangSmith **Compare** 기능으로 실험 결과 나란히 비교
- 성능, 비용, 속도를 종합 고려한 모델 선택 기준 이해

## 사전 지식

- 03번 노트북에서 LangSmith `evaluate()` 사용 경험
- LangSmith 데이터셋 생성 방법

## 왜 모델 비교가 필요한가?

새 모델이 출시될 때마다 "우리 시스템에서도 더 좋을까?"라는 질문이 생겨요. 답은 벤치마크가 아닌 **우리 데이터셋으로 직접 평가**해야 알 수 있습니다.

| 고려 사항 | GPT-4o-mini | GPT-3.5-turbo |
|---------|------------|--------------|
| 성능 | 높음 | 중간 |
| 비용 | 낮음 | 매우 낮음 |
| 속도 | 빠름 | 매우 빠름 |
| 특징 | 균형 잡힌 성능 | 단순 태스크에 적합 |

> **핵심**: 동일한 데이터셋으로 같은 평가자를 사용해야 공정한 비교가 가능해요.

---

## 환경 설정

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(f"LANGCHAIN_API_KEY: {'✅ 설정됨' if os.getenv('LANGCHAIN_API_KEY') else '❌ 미설정'}")


---

## 여러 모델로 RAG 시스템 구축

`create_rag_function(model_name)` 팩토리 함수를 만들어서 모델만 바꾸고 나머지 설정은 동일하게 유지해요. 공정한 비교를 위해 임베딩, 청크 크기, 프롬프트는 모두 같게 합니다.

In [ ]:
# ---------------------------------------------------
# 여러 모델로 RAG 시스템 구축 (팩토리 패턴 활용)
# ---------------------------------------------------
# ============================================================
# TODO: create_rag_function(model_name)으로 두 개의 RAG 함수를 만드세요
# 힌트: rag_gpt4_mini = create_rag_function("gpt-4o-mini"), rag_gpt35 = create_rag_function("gpt-3.5-turbo")
# 예상 결과: "여러 모델로 RAG 시스템 구축 완료" 출력
# ============================================================

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1단계: 공통 구성 요소 초기화 (모든 모델이 공유)
loader = PyMuPDFLoader("data/attention_paper.pdf")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

# 임베딩과 벡터 DB는 동일하게 유지 → 모델만 다르게 테스트
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever()

prompt = PromptTemplate.from_template(
    """Answer based on context.

Context: {context}
Question: {question}
Answer:"""
)

# 2단계: 모델별 RAG 함수 생성 팩토리


---

## 평가 실행 및 비교

각 모델을 **같은 데이터셋**으로 평가해요. `experiment_prefix`를 동일하게 설정하면 LangSmith가 같은 실험 그룹으로 묶어줍니다.

평가가 완료되면 LangSmith 웹 대시보드에서:
1. **Datasets** → 해당 데이터셋 선택
2. **Experiments** 탭에서 두 실험을 체크
3. **Compare** 버튼 클릭 → 나란히 비교 가능

In [ ]:
# ---------------------------------------------------
# 두 모델을 동일한 데이터셋으로 평가하여 LangSmith에 기록
# ---------------------------------------------------
# ============================================================
# TODO: rag_gpt4_mini와 rag_gpt35를 같은 데이터셋으로 각각 evaluate()를 실행하세요
# 힌트: experiment_prefix를 동일하게 설정하면 LangSmith가 같은 그룹으로 묶어줌
# 예상 결과: 두 실험의 URL 출력 및 LangSmith Compare 안내
# ============================================================

from langsmith import Client
from langsmith import utils as ls_utils
from langsmith.evaluation import evaluate

client = Client()
dataset_name = "transformer-qa-model-comparison"

# 데이터셋 준비
qa_pairs = [
    {"question": "What is the Transformer architecture?"},
    {"question": "How does self-attention work?"},
    {"question": "What are the advantages of Transformers?"},
]



---

## 정리

### 모델 비교 평가의 핵심 원칙

1. **동일한 데이터셋**: 같은 질문으로 비교해야 공정해요
2. **동일한 평가자**: 평가 기준이 같아야 결과를 신뢰할 수 있어요
3. **동일한 환경**: 임베딩, 청크 크기, 검색 파라미터를 고정하고 LLM만 바꿔요
4. **충분한 샘플**: 3~5개로는 통계적으로 신뢰하기 어렵고, 20개 이상 권장해요

### 비교 시 체크리스트

- 성능 점수뿐 아니라 **Latency(응답 속도)**도 비교
- **Token 사용량**으로 비용 추정
- 특정 질문 유형(단순/추론/멀티컨텍스트)별 성능 차이 확인
- 실패 케이스 유형 비교 (어떤 질문에서 실패하는가)

### 다음 노트북 예고

지금까지는 모두 **Offline 평가** — 사전에 준비된 데이터셋으로 평가 — 였어요. 마지막 노트북에서는 **Online 평가** — 프로덕션에서 실제 사용자 요청을 실시간 모니터링 — 의 개념과 방법을 배워볼게요.